# Markov Chain Monte Carlo

It is a method to draw samples from a distribution you cannot compute directly.

For instance, do not know the exact form of this posterior
$$
p(w \mid X, y) \propto p(y \mid X, w)p(w)
$$
Thus, cannot sample directly from the posterior, but can evaluate it up to a contant.

The MCMC algorithm lets you generate samples
$$
w^{(1)}, w^{(2)}, \cdots, w^{(S)} \sim p(w \mid X, y)
$$

---

The most common MCMC algorithm is Metropolis–Hastings. 
At iteration t:
1. Current state $w$
2. Propose a new candidate: 
$$
w' \sim \mathcal{N}(w, \sigma^2 I)
$$
3. Compute acceptance ratio:
$$
r = \frac{p(y \mid X, w')p(w')}{p(y \mid X, w)p(w)}
$$
4. Accept with probability 
$$
\min(1, r)
$$

---

## Application to Bayesian logistic regression

Likelihood:
$$
p(y \mid X, y) = \prod_i \sigma(x_i ^\top w)^{y_i}(1 - \sigma(x_i ^\top w))^{(1 - y_i)}
$$
Prior
$$
p(w) = \mathcal{N}(0, \alpha^{-1} I)
$$

In [2]:
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=100, 
    n_features=2, n_informative=2, 
    n_targets=1, 
    bias=0.0, 
    effective_rank=None, 
    tail_strength=0.5, 
    noise=0.0, 
    shuffle=True, 
    coef=False, random_state=42
)
X[:5], y[:5]

(array([[-1.60748323,  0.18463386],
        [-0.26465683,  2.72016917],
        [ 1.46564877, -0.2257763 ],
        [ 1.86577451,  0.47383292],
        [-1.0708925 ,  0.48247242]]),
 array([-127.35915354,  178.28131748,  111.86727647,  198.79808722,
         -58.21718166]))

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [4]:
import numpy as np 


def log_posterior(w, X, y, alpha):
    # Core log-posterior
    z = X @ w
    log_lik = np.sum(
        y * (-np.logaddexp(0, -z)) +
        (1 - y) * (-np.logaddexp(0, z))
    )
    log_prior = -0.5 * alpha * np.dot(w, w)
    return log_lik + log_prior

In [5]:
def metropolis_hastings(X, y, alpha=1.0, n_samples=5000, step_size=0.05):
    _, D = X.shape

    samples = []
    w = np.zeros(D)

    current_log_post = log_posterior(w, X, y, alpha)

    accept_count = 0

    for _ in range(n_samples):
        # Propose new w
        w_prop = w + step_size * np.random.randn(D)

        prop_log_post = log_posterior(w_prop, X, y, alpha)

        # Acceptance probability
        log_accept_ratio = prop_log_post - current_log_post

        if np.log(np.random.rand()) < log_accept_ratio:
            w = w_prop
            current_log_post = prop_log_post
            accept_count += 1

        samples.append(w.copy())

    print("Acceptance rate:", accept_count / n_samples)

    return np.array(samples)

In [6]:
samples = metropolis_hastings(X, y.ravel(), alpha=1.0)

# Remove burn-in
samples = samples[1000:]

# Posterior mean
w_mean = samples.mean(axis=0)

# Posterior covariance
w_cov = np.cov(samples.T)

Acceptance rate: 0.4908


In [8]:
def predict_mcmc(X_star, samples):
    logits = X_star @ samples.T
    probs = 1 / (1 + np.exp(-logits))

    mean = probs.mean(axis=1)
    std = probs.std(axis=1)

    lower = np.percentile(probs, 2.5, axis=1)
    upper = np.percentile(probs, 97.5, axis=1)

    return mean, std, np.vstack((lower, upper)).T

In [10]:
mean, std, ci = predict_mcmc(X, samples)

print("Predicted mean:\n", mean[:5])
print("\nPredictive std:\n", std[:5])
print("\n95% CI:\n", ci[:5])

Predicted mean:
 [2.32880599e-12 1.00000000e+00 1.00000000e+00 1.00000000e+00
 9.34893424e-07]

Predictive std:
 [1.54836751e-11 1.03451413e-14 1.33617954e-11 0.00000000e+00
 4.42007996e-06]

95% CI:
 [[2.34177885e-49 1.42349741e-11]
 [1.00000000e+00 1.00000000e+00]
 [1.00000000e+00 1.00000000e+00]
 [1.00000000e+00 1.00000000e+00]
 [4.62519670e-22 1.07312900e-05]]
